# PHASE 1: SET UP DEVELOPMENT ENVIRONMENT: INSTALLATIONS - IMPORTS

**Cell 1: Core Installations (GCP)**

In [ ]:
# INSTALLATIONS
%pip install --upgrade google-cloud-aiplatform

In [ ]:
%pip install --upgrade vertexai

In [ ]:
%pip install --upgrade google-genai

**LangChain Installations**

In [ ]:
# INSTALLATION: LANGCHAIN
%pip install --upgrade langchain langchain-core langchain-classic langchain-community
%pip install --upgrade langchain-text-splitters

**Google LangChain Integration**

In [ ]:
%pip install --upgrade langchain-google-community
%pip install --upgrade langchain-google-vertexai
%pip install --upgrade langchain-google-genai

**Utility and PDF Libraries**

In [ ]:
# Install: Utilities libraries needed for Q&A Search

# Dependencies required by Unstructured PDF loader
!sudo apt -y -qq install tesseract-ocr libtesseract-dev
!sudo apt-get -y -qq install poppler-utils

# %pip install --user --upgrade unstructured pdf2image pytesseract pdfminer.six
# %pip install --user --upgrade pillow-heif opencv-python unstructured-inference pikepdf pypdf

%pip install --user --upgrade unstructured pdf2image pytesseract pdfminer.six unstructured_pytesseract
%pip install --user --upgrade pillow-heif opencv-python unstructured-inference pikepdf pypdf pi_heif

**Authentication and Project Setup**

In [ ]:
import sys

from google.colab import auth
from google.cloud import storage

# 1. Authenticate user
# auth.authenticate_user()

# Authenticate when running inside Google Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()

# 2. Set project ID
PROJECT_ID = 'your-gcp-project-id'      # XYZ name - not real name - only for illustration
!gcloud config set project {PROJECT_ID}

# 3. Initialize client
storage_client = storage.Client(project=PROJECT_ID)
print(f"Authenticated with project: {storage_client.project}")

# Check host project
!gcloud config get-value project

**Vertex AI Base Imports**

In [ ]:
# ---- IMPORT FROM GCP: VERTEX AI ----

from google.cloud import aiplatform
import vertexai

# CHANGE: The import path for Namespace and NumericNamespace remains the same.
# These classes are still located in the matching_engine submodule of
# google-cloud-aiplatform. They are used for filtered vector similarity search
# (e.g., restricting results by string tags or numeric ranges).
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import (
    Namespace,
    NumericNamespace,
)

**Build System Development Environment**

In [ ]:
# BUILD SYSTEM DEVELOPMENT ENVIRONMENT

# Already set PROJECT_ID: PROJECT_ID = "ai-dl-qasearch"

REGION = "us-central1"
BUCKET_NAME = "your-gcs-bucket-name" # xyz name - nOT real name - used only for illustration
folder_prefix = "documents/pdfs/"

BUCKET_URI = f"gs://{BUCKET_NAME}/{folder_prefix}"

**Data Readiness (Commented)**

In [ ]:
# ALL PDFS ARE ALREADY IN THE BUCKET - DON'T DO ANYTHING HERE

""" COMMENT ALL

!gcloud storage cp -r gs://github-repo/documents/google-research-pdfs/* {BUCKET_URI}

"""

**LangChain Modern Retrieval Imports**

In [ ]:
# NEW: Import from langchain_classic (the package that now hosts all chain code):
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# NEW:
from langchain_google_community import GCSDirectoryLoader

# NEW:
from langchain_core.prompts import PromptTemplate

# CHANGE (additional import): ChatPromptTemplate is imported from langchain_core.prompts
# because create_stuff_documents_chain (the replacement for RetrievalQA) requires a
# ChatPromptTemplate rather than a plain PromptTemplate. This is needed to build
# the modern retrieval chain.
from langchain_core.prompts import ChatPromptTemplate

# NEW (single correct import):
from langchain_text_splitters import RecursiveCharacterTextSplitter

# CHANGE: PyPDFLoader import from langchain_community.document_loaders is correct
# and remains unchanged. This is the proper location in the reorganized ecosystem.

**GCS Loaders**

In [ ]:
# Need these LangChain Loaders to load the documents (in the buckets) into GCP Blobs
# Documents must be GCP blobs to be ready for embedding to create vector database
from langchain_community.document_loaders import GCSDirectoryLoader
from langchain_community.document_loaders import GCSFileLoader

**Vertex AI & LangChain API Imports**

In [ ]:
# ---- IMPORT FROM GCP: VERTEX AI & LANGCHAIN API ----

# CHANGE: These imports from langchain_google_vertexai remain correct and unchanged.
# This is the official Google partner package for Vertex AI integration with LangChain.
# - VertexAI: LLM wrapper for Vertex AI text models (e.g., text-bison, gemini)
# - VertexAIEmbeddings: Embedding model wrapper (e.g., text-embedding-005)
# - VectorSearchVectorStore: LangChain vector store backed by Vertex AI Vector Search
# - VectorSearchVectorStoreDatastore: Variant that uses Datastore for document storage
from langchain_google_vertexai import VertexAI
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_google_vertexai import (
    VectorSearchVectorStore,
    VectorSearchVectorStoreDatastore,
)

**Other Utility Imports**

In [ ]:
# ---- IMPORT OTHERS ----

import textwrap

**Version Verification**

In [ ]:
import importlib.metadata

# Check versions of the main sw platforms of the system

# GCP aiplatform & vertex ai
import vertexai
from google.cloud import aiplatform

print(f"aiplatform SDK version: {aiplatform.__version__}")
print(f"Vertex AI SDK version: {vertexai.__version__}")

# LangChain, langchain-core, langchain-community
import langchain
print(f"LangChain version: {langchain.__version__}")

from langchain_core import __version__ as langchain_core_version
print(f"langchain-core version: {langchain_core_version}")

# from langchain_classic import __version__ as langchain_classic_version
# print(f"langchain-classic version: {langchain_classic_version}")
# Fix: Use importlib.metadata for langchain_classic as well, as it might have a similar issue.
langchain_classic_version = importlib.metadata.version('langchain-classic')
print(f"langchain-classic version: {langchain_classic_version}")

from langchain_community import __version__ as langchain_community_version
print(f"langchain-community version: {langchain_community_version}")

from langchain_google_community import __version__ as langchain_google_community_version
print(f"langchain-google-community version: {langchain_google_community_version}")

# Fix: Use importlib.metadata to get the version for langchain_google_vertexai
langchain_google_vertexai_version = importlib.metadata.version('langchain-google-vertexai')
print(f"langchain-google-vertexai version: {langchain_google_vertexai_version}")

**AI Platform Initialization**

In [ ]:
# =========================================================================
# INITIALIZE GCP AI PLATFORM FOR AI SW SYSTEM
# =========================================================================

# Initialize the system (the current AI software application)
# This AI software application is associated with the GCP project as declared
# This AI software application runs at the specified GCP region
# This AI software application uses the specified GCS bucket

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

**Text Embedding Model Setup**

In [ ]:
# =========================================================================
# DEFINE TEXT EMBEDDING MODEL CREDENTIALS & CREATE IT
# =========================================================================

# The number of dimensions for text-embedding-005 is 768.
# CHANGE: Updated the comment. The original comment referenced "textembedding-gecko@003"
# but the code already correctly uses "text-embedding-005". The text-embedding-005 model
# is Google's latest text embedding model, which replaced the older gecko models.
# It produces 768-dimensional vectors, same as gecko@003.
DIMENSIONS = 768

# Index Constants
# DISPLAY_NAME_ID: Choose a name specific for your project
# DEPLOYED_INDEX_ID: Choose a name specific for your project
DISPLAY_NAME_ID = "group2_nutrition_index"       # XYZ name - not real name - only for illustration
DEPLOYED_INDEX_ID = "group2_nutrition_endpoint"   # XYZ name - not real name - only for illustration

# Create text embedding model based on GCP Vertex AI: text-embedding-005
embedding_model = VertexAIEmbeddings(model_name="text-embedding-005", project=PROJECT_ID)

# CLEANUP / MANAGEMENT OPERATIONS

# UNDEPLOY ALL INDICES

# DELETE ALL INDICES & ENDPOINTS

**List Existing Resources**

In [ ]:
list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

print("\n\n")

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print(list_end_points)

# SPECIAL CASE: ONLY ONE INDEX AND ONE ENDPOINT THAT HAS BEEN DEPLOYED

**Special Case (Single Resource)**

In [ ]:
# SPECIAL CASE: ONLY ONE INDEX AND ONE ENDPOINT THAT HAS BEEN DEPLOYED

my_index = aiplatform.MatchingEngineIndex.list()[0]
my_endpoint = aiplatform.MatchingEngineIndexEndpoint.list()[0]
deployed_index_id = my_endpoint.deployed_indexes[0].id

# =========================================================================
# GET & SET index_id and end_point_id
# =========================================================================
# These variables are set for convenience in subsequent code.

# INDEX NAME and INDEX ID
index_id = my_index.name
print(f"INDEX NAME: {index_id}")

# DEPLOYED INDEX ID (DEPLOYED_INDEX_ID = "adta5770_qasearch_endpoint_id")
print(f"DEPLOYED INDEX ID: {deployed_index_id}")

end_point_id = my_endpoint.name
print(f"end_point_id: {end_point_id}")

# GENERAL CASE: MULTIPLE INDEXES AND ENDPOINTS HAVE BEEN DEPLOYED
**Deleting Multiple Resources (General Case)**

In [ ]:
# GENERAL CASE: MULTIPLE INDEXES AND ENDPOINTS HAVE BEEN DEPLOYED

# UNDEPLOY INDEXES and DELETE ENDPOINTS
del_index_endpoint_1 = aiplatform.MatchingEngineIndexEndpoint("your-index-endpoint-id")

# Un-deploy indexes and delete end points
del_index_endpoint_1.undeploy_all()

# and delete end points
del_index_endpoint_1.delete()

**Deleting Indices**

In [ ]:
# DELETE INDICES

# Get the list of indexes that have been created for the vector search system
list_indexes = aiplatform.MatchingEngineIndex.list()
print(f"List of indexes: {list_indexes}")

del_index_1 = aiplatform.MatchingEngineIndex("7978190511561244672")
# ... Continue to get all indexes

del_index_1.delete()
# ... Continue until deleting all indexes

# Verify that all indexes have been deleted --> The list should be empty: Nothing is printed out
list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

###============================= AT THIS POINT:
# All indexes have been un-deployed and deleted.
# All end points have been deleted

**Final Check**

In [ ]:
# DISPLAY LIST OF ENDPOINTS AND INDEXES TO CHECK

list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

print("\n\n")

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print(list_end_points)

# **PHASE 2:** PROCESS DOCUMENTS

MAKE THEM READY FOR EMBEDDING AND VECTORIZING

**CHUNK DOCUMENTS --> CHUNKS**

This step ingests and parse PDF documents, split them, generate embeddings and
corpus used as dataset is a sample of Google published research papers across
productivity etc.

**LOAD PDF files in the gcs bucket into a KNOWLEDGE BASE**

NOTES:

--) In GCS: Each document/file is a "blob" (SQL: Blob is a collection of text)

--) Knowledge base: named as "documents_from_blob"

--) Knowledge base is a storage/data strcuture to store the documents

NOTES: It can take 5 minutes or more

**Load and Process PDFs**

In [ ]:
# --- Initialization ---
# Install missing package if not already present
%pip install unstructured --force-reinstall

print(f"Processing documents from gs://{BUCKET_URI}")

bucket = storage_client.bucket(BUCKET_NAME)

# --- Load Documents ---
all_documents = []
blobs = bucket.list_blobs(prefix=folder_prefix) # List blobs matching the prefix

for blob in blobs:
    print(str(blob))

    # Skip directories/folders if represented as blobs, and ensure it's a PDF
    if blob.name.endswith("/") or not blob.name.lower().endswith(".pdf"):
        continue

    print(f"  Loading document: {blob.name}")

    # Use GCSFileLoader for each PDF blob
    loader = GCSFileLoader(
        project_name=PROJECT_ID, bucket=BUCKET_NAME, blob=blob.name
    )

    # Load documents (GCSFileLoader often returns one Document per page)
    # VIP NOTES: Loader.load() only load one file at a time because it is "GCSFileLoader"
    documents_from_blob = loader.load()

    #==================== NEW NEW NEW

    # --- Metadata Enhancement (similar to original logic) ---
    # Derive document name from the blob name
    document_name = blob.name.split("/")[-1]

    # Derive doc source prefix and suffix to match original logic
    # Note: The original code's 'source' derivation seemed to point to the
    #       *folder* rather than the file. This replicates that.
    #       A more common approach might be to use the blob's full gs:// path
    #       as the source.
    doc_source_prefix = f"gs://{BUCKET_NAME}"
    # Get the directory path containing the blob
    doc_source_suffix = "/".join(blob.name.split("/")[0:-1])
    source = f"{doc_source_prefix}/{doc_source_suffix}" # Folder path as source

    # VIP NOTES: Only one document existed in documents_from_blob (GCSFileLoader is used)
    for document in documents_from_blob:
        # Add derived metadata to each document (page)
        document.metadata["source"] = source
        document.metadata["document_name"] = document_name

    # GCSFileLoader might add other useful metadata like 'page' automatically
    # NOW: Add this document into the list all_documents
    all_documents.extend(documents_from_blob)

# The 'all_documents' list now contains LangChain Document objects,
# likely one per page from all loaded PDFs.
print(f"# of document pages loaded (pre-chunking) = {len(all_documents)}")

# CHUNK DOCUMENTS

Split the documents to smaller chunks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)
doc_splits = text_splitter.split_documents(all_documents)

# add chunk number to metadata
for idx, split in enumerate(doc_splits):
    split.metadata["chunk"] = idx

print(f"# of documents = {len(doc_splits)}")

# PHASE 3

CREATE AN EMPTY VECTOR SEARCH INDEX

In [ ]:
# CREATE AN EMPTY VECTOR SEARCH INDEX

my_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=DISPLAY_NAME_ID,
    dimensions=DIMENSIONS,
    approximate_neighbors_count=150,
    distance_measure_type="DOT_PRODUCT_DISTANCE",
    index_update_method="STREAM_UPDATE", # allowed values BATCH_UPDATE, STREAM UPDATE
    leaf_node_embedding_count=500
)

if my_index:
  print(my_index.name)

list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

CREATE A PUBLIC ENDPOINT

In [ ]:
# CREATE A PUBLIC ENDPOINT

# create a public endpoint for the vector search index
my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"{DISPLAY_NAME_ID}-endpoint",
    public_endpoint_enabled=True
)

print(my_index_endpoint.name)

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print(list_end_points)

DEPLOY THE EMPTY VECTOR SEARCH INDEX TO THE ENDPOINT

MAKE IT READY FOR LOADING VECTORIZED EMBEDDING DATA TO THE INDEX

In [ ]:
my_index_endpoint = my_index_endpoint.deploy_index(
    index=my_index,
    deployed_index_id=DEPLOYED_INDEX_ID
)

my_index_endpoint.deployed_indexes

# PHASE 4

**Configure Matching Engine/Vector Search as Vector Store**

--) Initialize Matching Engine/Vector Search vector store with text embeddings mode

SET UP to access existing Matching Engine/Vector Search index (GCP: Vector Store)

**Vector Store Variables**

In [ ]:
# Define credentials
# VSVDB: Vector Search Vector Database (Vector Database: GCP Vector Store)

VSVDB_REGION = "us-central1"

# XYZ name - not real name - only for illustration
VSVDB_INDEX_NAME = "your-vector-search-index-name" # @param {type:"string"} - GCP components

# XYZ name - not real name - only for illustration
VSVDB_EMBEDDING_DIR = "your-gcs-embeddings-bucket-name" # @param {type:"string"} - GCP components

VSVDB_DIMENSIONS = 768 # when using Vertex AI text-embedding-005

Create a GCS bucket to store vector search index
VIP NOTES:

--) if an existing bucket has been created before for previous code runs, MUST delete it first

--) CHECK: VSVDB_EMBEDDING_DIR = "..." exists in GCP Sotrage of the project?

If it exists --> DELETE it

# IMPORTANT CODE: DON'T MISS

**Create Embedded Directory Bucket**

In [ ]:
# Create a GCS bucket to store the Matching Engine/Vector Search index

#! set -x && gsutil mb -p $PROJECT_ID -l us-central1 gs://$VSVDB_EMBEDDING_DIR

# PHASE 5
**VIP NOTES: CONFIGURE INDEX AS VECTOR STORE**

--) Create a vector database: Vector Store

--) Creating the vector database by configuring the index as an instance of GCP: Vector Store

**VIP NOTES:**

--) Before creating the vector store using the index, MUST DEPLOY the index

**Initialize Vector Store**

In [ ]:
# Embeddings API integrated with LangChain
# Create a Vector Search vector database, vsvectordb
# It is actually a GCP Vertex AI vector store
# initialize the vector store

vsvectordb = VectorSearchVectorStore.from_components(
    project_id=PROJECT_ID,
    region=VSVDB_REGION,
    gcs_bucket_name=f"gs://{VSVDB_EMBEDDING_DIR}".split("/")[2],
    embedding=embedding_model, # embedding_model = VertexAIEmbeddings(model_name="text-embedding-005")
    index_id=my_index.name,
    endpoint_id=my_index_endpoint.name,
    stream_update=True,
)

# PHASE 6

**Add documents as embeddings in Matching Engine as index**

The document chunks are transformed as embeddings (vectors) using Vertex AI Embeddings API and added to the index with streaming index update. With Streaming Updates, you can update and query your index within a few seconds.

The original document text is stored on Cloud Storage bucket had referenced by id.

**Add embeddings to the vector store**

**NOTE:**
Depending on the volume and size of documents, this step may take time.

**Add Chunked Texts to DB**

In [ ]:
# Store docs as embeddings in Matching Engine/Vector Search index
# It may take a while since API is rate limited
# The Vertex AI embedding model has a limit of 250 instances per prediction request.
# To avoid the '400 INVALID_ARGUMENT' error, we need to batch the texts.

texts = [doc.page_content for doc in doc_splits]
metadatas = [doc.metadata for doc in doc_splits]

# Define a batch size, slightly less than the limit (e.g., 200 or 250)
#batch_size = 100 # Reduced further to ensure total token count per batch is strictly < 20000

#for i in range(0, len(texts), batch_size):
    #batch_texts = texts[i : i + batch_size]
    #batch_metadatas = metadatas[i : i + batch_size]
    #print(f"Adding batch {i//batch_size + 1}/{(len(texts) + batch_size - 1)//batch_size} (size: {len(batch_texts)})... ")
    #vsvectordb.add_texts(texts=batch_texts, metadatas=batch_metadatas)

#print("All texts added successfully.")


import time

batch_size = 50
# Update this when re-running
start_index = 0

for i in range(start_index, len(texts), batch_size):
    batch_texts = texts[i : i + batch_size]
    batch_metadatas = metadatas[i : i + batch_size]
    batch_ids = [str(j) for j in range(i, i + len(batch_texts))]

    print(f"Adding batch {i//batch_size + 1} (size: {len(batch_texts)})... ")

    success = False
    retries = 0 # Keep track of network failures

    while not success:
        try:
            vsvectordb.add_texts(
                texts=batch_texts,
                metadatas=batch_metadatas,
                ids=batch_ids
            )
            success = True
            retries = 0 # Reset retries on success
            time.sleep(3)

        except Exception as e:
            error_str = str(e)

            # Catch Vertex AI Quota Limits
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                print("⏳ Token limit reached! Pausing for 60 seconds...")
                time.sleep(60)

            # NEW: Catch Google Cloud Storage Network Drops (503, 502, 500)
            elif "503" in error_str or "500" in error_str or "502" in error_str:
                retries += 1
                if retries > 5:
                    print(f"❌ Network failed 5 times in a row on batch {i//batch_size + 1}. Crashing.")
                    raise e
                print(f"🔌 Network hiccup (503). Retrying batch in 10 seconds... (Attempt {retries}/5)")
                time.sleep(10)

            # If it's a completely different error, crash so you can see it
            else:
                raise e

print("All texts added successfully.")


# SYSTEM CHAIN SETUP (Phase 7/8)

**Setup Gemini LLM**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro", # Changed to gemini-pro for wider availability
    project=PROJECT_ID, # Ensure project ID is passed for Vertex AI access
    location="us-central1",    # Changed to us-central1 as an alternative region
    max_output_tokens=8192,
    temperature=0.2,
    top_p=0.8,
    top_k=40,
    verbose=True,
)

## Configure an instance of RetrievalQA

**VIP NOTES:**

--) A Vertex AI vector search index, i.e., a vector database, set up as a Vertex AI vector store can be configured as a RetrievalQA.

--) RetrievalQA is a Vertex AI vector search tool specifically used for Q&A Search

First, need to configure a retriever

--) Retriever is used to retrieve response from the vector search index (or vector store)

**Retriever Config**

In [ ]:
# Create chain to answer questions

# Number of responses that the system tries to search for
NUMBER_OF_RESULTS = 10

# The distance (similarity search) used as a threshold to search for responses
SEARCH_DISTANCE_THRESHOLD = 0.6

# Expose index to the retriever
retriever = vsvectordb.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": NUMBER_OF_RESULTS
    },
    filters=None,
)

**Prompt Template**

In [ ]:
"""### Customize a prompt template"""

# CHANGE (PHASE 8 FIX): The prompt template variable names must match what
# create_retrieval_chain and create_stuff_documents_chain expect:
#   - {input}   : the user's question (was {question} in the old RetrievalQA template)
#   - {context} : the retrieved document chunks (remains the same)
# Using {question} would cause a KeyError because the new chain passes the user query
# as "input", not "question". The template is also wrapped in ChatPromptTemplate.from_template()
# because create_stuff_documents_chain requires a ChatPromptTemplate (not a plain PromptTemplate).
# A plain PromptTemplate causes a ValidationError.
template = """SYSTEM: You are an intelligent assistant helping users with their questions on nutrition and dietary information.

Question: {input}

Strictly Use ONLY the following pieces of context to answer the question at the end. Think step-by-step and then answer.

Do not try to make up an answer:
  - If the answer to the question cannot be determined from the context alone, say "I cannot determine the answer from the context."
  - If the context is empty, just say "I do not know the answer to that."

=============
{context}
=============

Question: {input}
Helpful Answer:"""

In [ ]:
# Create the ChatPromptTemplate from the template string
# CHANGE: create_stuff_documents_chain requires ChatPromptTemplate, not PromptTemplate.
# ChatPromptTemplate properly formats the prompt as a chat message, which is required
# by modern LangChain chain functions.
qa_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Create the document-combining chain (replaces chain_type="stuff" in RetrievalQA)
combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)

In [ ]:
# Create the full retrieval chain (replaces RetrievalQA.from_chain_type())
retrieval_qa_chain = create_retrieval_chain(retriever, combine_docs_chain)

**Debug Settings**

In [ ]:
from langchain_core.globals import set_verbose, set_debug
set_verbose(True)

# Uncomment the following line for even more detailed debug output:
# set_debug(True)

# UTILITY FUNCTIONS
**Output Formatting**

In [ ]:
"""### Utility function used to format the response"""

def formatter(result):
    print(f"Query: {result['input']}")
    print("." * 80)
    if "context" in result.keys():
        for idx, ref in enumerate(result["context"]):
            print("-" * 80)
            print(f"REFERENCE #{idx}")
            print("-" * 80)
            if "score" in ref.metadata:
                print(f"Matching Score: {ref.metadata['score']}")
            if "source" in ref.metadata:
                print(f"Document Source: {ref.metadata['source']}")
            if "document_name" in ref.metadata:
                print(f"Document Name: {ref.metadata['document_name']}")
            print("." * 80)
            print(f"Content: \n{wrap(ref.page_content)}")
        print("." * 80)
        print(f"Response: {wrap(result['answer'])}")
        print("." * 80)

**Text Wrapping**

In [ ]:
def wrap(s):
    return "\n".join(textwrap.wrap(s, width=120, break_long_words=False))

**Querying Function**

In [ ]:
def ask(
    query,
    k=NUMBER_OF_RESULTS,
    search_distance=SEARCH_DISTANCE_THRESHOLD,
    filters=None,
):
    retriever.search_kwargs["k"] = k

    if filters:
        ns_filters = [Namespace(name=filters["namespace"],
                                allow_tokens=filters["allow_list"])]
        retriever.search_kwargs["filter"] = ns_filters
    else:
        # Clear any previously set filters
        retriever.search_kwargs.pop("filter", None)

    # CHANGE: Use .invoke() instead of calling the chain directly.
    # The old pattern chain({"query": ...}) is deprecated in LangChain v1.
    # .invoke() is the standard LCEL interface for running chains.
    result = retrieval_qa_chain.invoke({"input": query})
    return formatter(result)

# A SIMPLE TEST TO VERIFY THE SYSTEM, END-TO-END, WORKS CORRECTLY
**Initial Test Query**

In [ ]:
ask("Who can register commitments in the NAF?")

# PHASE 9: TEST Q&A SEARCH SYSTEM

### Q&A SEARCH with FILTERS

**Test Query with Metadata Filters**

In [ ]:
filters = {
    "namespace": "document_name",
    "allow_list": ["2022_Global_Nutrition_Report.pdf"],
}
ask("Who can register commitments in the NAF?", filters=filters)

In [ ]:
#ask("What affects cardiovascular health")

In [ ]:
#ask("What time is graduation for the College of Science?")

In [ ]:
#ask("Which public database does the method rely on to source its actual nutritional values?")

In [ ]:
#ask("According to the study's analysis of Reddit posts, what major global event coincided with a noticeable plateau of higher-calorie food posts?")

In [ ]:
#ask("What foods are highest in protein?")

In [ ]:
#ask("How do governments fufill their responsibility to safeguard their populations’ nutrition")

In [ ]:
#ask("What is Unified Meal Representation Learning (UMRL)? ")

In [ ]:
#ask("What two seed treatment methods have been shown to significantly enhance the nutritional profile and germination rate of alfalfa sprouts?")

In [ ]:
#ask("In which population has Melissa phospholipid supplementation demonstrated measurable improvements in sleep quality?")

In [ ]:
--ask("What three chronic health conditions have been strongly linked to excessive sugar consumption in adults?")

In [ ]:
#ask("What two dietary changes does the American Heart Association recommend to improve long-term cardiovascular health outcomes?")

In [ ]:
#ask("According to the International Society of Sports Nutrition, how many grams of protein per kilogram of body weight should athletes consume daily to support muscle repair?")

In [ ]:
#ask("What technology enables AI models to accurately estimate the caloric and macronutrient content of meals from photographs?")

In [ ]:
#ask("Why does managing nutritional support for diabetic patients present significant challenges?")

In [ ]:
#ask("To what application can deep reinforcement learning frameworks be applied in intensive care units?")

In [ ]:
#ask("What is the primary purpose of national dietary surveys in the context of nutritional research?")

In [ ]:
#ask("What two types of data does systems biology integrate to design personalized nutrition plans tailored to individual metabolic profiles?")

In [ ]:
#ask("What is NAF and its contribution to the success of the Tokyo N4G Summit?")

In [ ]:
#ask("What is a Labubu?")

# PHASE 10

##### UNDEPLOY INDEX AND DELETE ALL INDEXES & ENDPOINTS

# CLEANUP / MANAGEMENT OPERATIONS

# UNDEPLOY ALL INDICES

# DELETE ALL INDICES & ENDPOINTS

**List Existing Resources**

In [ ]:
list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

print("\n\n")

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print(list_end_points)

# SPECIAL CASE: ONLY ONE INDEX AND ONE ENDPOINT THAT HAS BEEN DEPLOYED

**Special Case (Single Resource)**

In [ ]:
# SPECIAL CASE: ONLY ONE INDEX AND ONE ENDPOINT THAT HAS BEEN DEPLOYED

my_index = aiplatform.MatchingEngineIndex.list()[0]
my_endpoint = aiplatform.MatchingEngineIndexEndpoint.list()[0]
deployed_index_id = my_endpoint.deployed_indexes[0].id

# =========================================================================
# GET & SET index_id and end_point_id
# =========================================================================
# These variables are set for convenience in subsequent code.

# INDEX NAME and INDEX ID
index_id = my_index.name
print(f"INDEX NAME: {index_id}")

# DEPLOYED INDEX ID (DEPLOYED_INDEX_ID = "adta5770_qasearch_endpoint_id")
print(f"DEPLOYED INDEX ID: {deployed_index_id}")

end_point_id = my_endpoint.name
print(f"end_point_id: {end_point_id}")

# GENERAL CASE: MULTIPLE INDEXES AND ENDPOINTS HAVE BEEN DEPLOYED
**Deleting Multiple Resources (General Case)**

In [ ]:
# GENERAL CASE: MULTIPLE INDEXES AND ENDPOINTS HAVE BEEN DEPLOYED

# UNDEPLOY INDEXES and DELETE ENDPOINTS
del_index_endpoint_1 = aiplatform.MatchingEngineIndexEndpoint("your-index-endpoint-id")

# Un-deploy indexes and delete end points
del_index_endpoint_1.undeploy_all()

# and delete end points
del_index_endpoint_1.delete()

**Deleting Indices**

In [ ]:
# DELETE INDICES

# Get the list of indexes that have been created for the vector search system
list_indexes = aiplatform.MatchingEngineIndex.list()
print(f"List of indexes: {list_indexes}")

del_index_1 = aiplatform.MatchingEngineIndex("3927906334183260160")
# ... Continue to get all indexes

del_index_1.delete()
# ... Continue until deleting all indexes

# Verify that all indexes have been deleted --> The list should be empty: Nothing is printed out
list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

###============================= AT THIS POINT:
# All indexes have been un-deployed and deleted.
# All end points have been deleted

**Final Check**

In [ ]:
# DISPLAY LIST OF ENDPOINTS AND INDEXES TO CHECK

list_indexes = aiplatform.MatchingEngineIndex.list()
print(list_indexes)

print("\n\n")

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print(list_end_points)